# 02-14 nnU-Net PHE with ICH teacher prior

Notebook nay dung ICH teacher da train o `02_13` de predict ICH tren PHE-SICH, sau do train PHE nnU-Net voi 2 channel:

```text
channel 0 = CT
channel 1 = ICH teacher prior mask
label     = manual PHE mask
```

Quan trong: notebook nay khong train lai ICH teacher. ICH teacher chi duoc load checkpoint va chay inference de tao prior.

In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
NNUNET_REPO = PROJECT_ROOT / 'nnUNet-master'

PHE_ROOT = PROJECT_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT'
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'
SPLIT_CSV = PROJECT_ROOT / 'outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation' / 'manifests' / '3dff_iph_phe_patient_split.csv'

ICH_TEACHER_ROOT = PROJECT_ROOT / 'outputs_02_13_nnunet_ich_teacher_external'
ICH_TEACHER_DATASET_NAME = 'Dataset113_ExternalICHTeacher'
ICH_TEACHER_CONFIGURATION = '3d_fullres'
ICH_TEACHER_TRAINER = 'nnUNetTrainer_250epochs'
ICH_TEACHER_PLANS = 'nnUNetPlans'
ICH_TEACHER_FOLD = 0
ICH_TEACHER_CHECKPOINT = 'checkpoint_best.pth'
ICH_TEACHER_MODEL_DIR = (
    ICH_TEACHER_ROOT / 'nnunet_workdir' / 'nnUNet_results' / ICH_TEACHER_DATASET_NAME /
    f'{ICH_TEACHER_TRAINER}__{ICH_TEACHER_PLANS}__{ICH_TEACHER_CONFIGURATION}'
)
ICH_TEACHER_CHECKPOINT_PATH = ICH_TEACHER_MODEL_DIR / f'fold_{ICH_TEACHER_FOLD}' / ICH_TEACHER_CHECKPOINT

OUTPUT_ROOT = PROJECT_ROOT / 'outputs_02_14_nnunet_phe_ich_teacher_prior'
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
NNUNET_RESULTS = NNUNET_BASE / 'nnUNet_results'

DATASET_ID = 114
DATASET_NAME = f'Dataset{DATASET_ID:03d}_PHESICH_PHE_ICHTeacherPrior'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

ICH_INPUT_DIR = OUTPUT_ROOT / 'ich_teacher_inputs_phe_sich'
ICH_PRIOR_MASK_DIR = OUTPUT_ROOT / 'ich_teacher_prior_masks'

for p in [OUTPUT_ROOT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, ICH_INPUT_DIR, ICH_PRIOR_MASK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

assert NNUNET_REPO.exists(), f'Missing nnU-Net repo: {NNUNET_REPO}'
assert PHE_IMAGE_DIR.exists(), f'Missing PHE image dir: {PHE_IMAGE_DIR}'
assert PHE_MASK_DIR.exists(), f'Missing PHE mask dir: {PHE_MASK_DIR}'
assert SPLIT_CSV.exists(), f'Missing split CSV: {SPLIT_CSV}'
assert ICH_TEACHER_CHECKPOINT_PATH.exists(), f'Missing ICH teacher checkpoint: {ICH_TEACHER_CHECKPOINT_PATH}'

print('Project root          :', PROJECT_ROOT)
print('nnU-Net repo          :', NNUNET_REPO)
print('PHE image dir         :', PHE_IMAGE_DIR)
print('PHE mask dir          :', PHE_MASK_DIR)
print('Split CSV            :', SPLIT_CSV)
print('ICH teacher model    :', ICH_TEACHER_MODEL_DIR)
print('ICH teacher ckpt     :', ICH_TEACHER_CHECKPOINT_PATH)
print('Output root          :', OUTPUT_ROOT)
print('Dataset              :', DATASET_NAME)
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])


Project root          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao
nnU-Net repo          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master
PHE image dir         : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-IDS\SubdatasetA_NIFIT\NIFIT\set
PHE mask dir          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-IDS\SubdatasetA_NIFIT\NIFIT\label
Split CSV            : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation\manifests\3dff_iph_phe_patient_split.csv
ICH teacher model    : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13_nnunet_ich_teacher_external\nnunet_workdir\nnUNet_results\Dataset113_ExternalICHTeacher\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
ICH teacher ckpt     : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13_nnunet_ich_teacher_external\nnunet_workdir\nnUNet_results\Dataset113_ExternalICHTeacher\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres\fold_0\checkpoint_best.pth
Output root          : D:\Thuy_Loi\Nam_3\CT_xuathuye

## 0. Dependency check

In [2]:
AUTO_INSTALL_MISSING_DEPS = False
INSTALL_NNUNET = False

def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

if INSTALL_NNUNET:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

ensure_import('SimpleITK')
ensure_import('nnunetv2')

import SimpleITK as sitk
import nnunetv2
print('SimpleITK OK')
print('nnunetv2 import OK:', nnunetv2.__file__)


SimpleITK OK
nnunetv2 import OK: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master\nnunetv2\__init__.py


## 1. Inspect split PHE-SICH

Dung lai split tu `02_10`/`02_12`: train 99, val 10, test 11.

In [3]:
split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
split_df.columns = [str(c).strip() for c in split_df.columns]

def make_nnunet_case_id(value, img_path=None):
    value = '' if value is None or pd.isna(value) else str(value).strip()
    if not value and img_path is not None:
        name = Path(str(img_path)).name
        value = name.replace('.nii.gz', '').replace('.nii', '')
    if value.lower().startswith('phe_'):
        return value
    if value.isdigit():
        return f'phe_{int(value):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in value).strip('_')
    return f'phe_{safe}' if not safe.lower().startswith('phe_') else safe

if 'patient_id' in split_df.columns:
    split_df['case_id'] = [make_nnunet_case_id(v, p) for v, p in zip(split_df['patient_id'], split_df['img_path'])]
elif 'scan_id' in split_df.columns:
    split_df['case_id'] = [make_nnunet_case_id(v, p) for v, p in zip(split_df['scan_id'], split_df['img_path'])]
else:
    split_df['case_id'] = [make_nnunet_case_id(None, p) for p in split_df['img_path']]

required_cols = {'case_id', 'split', 'img_path', 'phe_mask_path'}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f'Missing columns in split CSV: {missing_cols}'
assert split_df['case_id'].is_unique, 'case_id must be unique.'

split_df['img_exists'] = split_df['img_path'].map(lambda x: Path(x).exists())
split_df['mask_exists'] = split_df['phe_mask_path'].map(lambda x: Path(x).exists())
assert split_df['img_exists'].all(), 'Some image paths are missing.'
assert split_df['mask_exists'].all(), 'Some PHE mask paths are missing.'

display(split_df.groupby('split').agg(
    cases=('case_id', 'count'),
    slices=('n_slices', 'sum'),
    phe_positive_slices=('phe_positive_slices', 'sum'),
    total_phe_ml=('phe_volume_ml', 'sum'),
))
display(split_df[['case_id', 'split', 'img_path', 'phe_mask_path']].head())
split_df.to_csv(OUTPUT_ROOT / '02_14_phe_sich_split_manifest.csv', index=False)
print('Total cases:', len(split_df))


,cases,slices,phe_positive_slices,total_phe_ml
split,,,,
test,11,329,62,31.534920
train,99,2983,560,355.115621
val,10,289,61,34.999343


,case_id,split,img_path,phe_mask_path
0,phe_0001,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
1,phe_0002,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
2,phe_0004,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
3,phe_0005,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
4,phe_0006,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...


Total cases: 120


## 2. Predict ICH prior on PHE-SICH

Cell nay chi load checkpoint ICH teacher tu `02_13` va predict mask ICH cho toan bo case PHE-SICH. Khong train lai ICH.

In [4]:
RUN_ICH_PRIOR_PREDICT = True
OVERWRITE_ICH_INPUTS = True
OVERWRITE_ICH_PRIORS = False
ICH_DISABLE_TTA = False

if OVERWRITE_ICH_INPUTS and ICH_INPUT_DIR.exists():
    shutil.rmtree(ICH_INPUT_DIR)
ICH_INPUT_DIR.mkdir(parents=True, exist_ok=True)
ICH_PRIOR_MASK_DIR.mkdir(parents=True, exist_ok=True)

input_records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    src = Path(row.img_path)
    dst = ICH_INPUT_DIR / f'{case_id}_0000.nii.gz'
    if OVERWRITE_ICH_INPUTS or not dst.exists():
        shutil.copy2(src, dst)
    input_records.append({'case_id': case_id, 'split': row.split, 'ich_teacher_input': str(dst)})

ich_input_df = pd.DataFrame(input_records)
ich_input_df.to_csv(OUTPUT_ROOT / '02_14_ich_teacher_input_manifest.csv', index=False)

expected_prior_paths = [ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz' for case_id in split_df['case_id'].astype(str)]
missing_priors = [p for p in expected_prior_paths if not p.exists()]
print('ICH teacher inputs:', len(list(ICH_INPUT_DIR.glob('*.nii.gz'))))
print('Existing ICH priors:', len(expected_prior_paths) - len(missing_priors), '/', len(expected_prior_paths))

if RUN_ICH_PRIOR_PREDICT and (OVERWRITE_ICH_PRIORS or missing_priors):
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('ICH teacher predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not ICH_DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(ICH_TEACHER_MODEL_DIR),
        use_folds=(ICH_TEACHER_FOLD,),
        checkpoint_name=ICH_TEACHER_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(ICH_INPUT_DIR),
        str(ICH_PRIOR_MASK_DIR),
        save_probabilities=False,
        overwrite=OVERWRITE_ICH_PRIORS,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
elif RUN_ICH_PRIOR_PREDICT:
    print('All ICH priors already exist. Set OVERWRITE_ICH_PRIORS=True to regenerate.')
else:
    print('Skipped ICH teacher inference. Set RUN_ICH_PRIOR_PREDICT=True to run.')

print('ICH prior dir:', ICH_PRIOR_MASK_DIR)


ICH teacher inputs: 120
Existing ICH priors: 0 / 120
ICH teacher predict device: cuda
There are 120 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 120 cases that I would like to predict
overwrite was set to False, so I am only working on cases that haven't been predicted yet. That's 120 cases.

Predicting phe_0001:
perform_everything_on_device: True


100%|██████████| 180/180 [01:45<00:00,  1.70it/s]


sending off prediction to background worker for resampling and export
done with phe_0001

Predicting phe_0002:
perform_everything_on_device: True


100%|██████████| 180/180 [01:44<00:00,  1.73it/s]


sending off prediction to background worker for resampling and export
done with phe_0002

Predicting phe_0004:
perform_everything_on_device: True


...

100%|██████████| 180/180 [01:43<00:00,  1.74it/s]


sending off prediction to background worker for resampling and export
done with phe_0166

Predicting phe_0167:
perform_everything_on_device: True


100%|██████████| 120/120 [01:08<00:00,  1.75it/s]


sending off prediction to background worker for resampling and export
done with phe_0167
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
ICH prior dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\ich_teacher_prior_masks


## 3. Quick inspect ICH prior

In [5]:
INSPECT_PRIORS = True
MAX_INSPECT = 8

if INSPECT_PRIORS:
    inspect_rows = []
    for row in split_df.head(MAX_INSPECT).itertuples(index=False):
        case_id = str(row.case_id)
        prior_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
        if not prior_path.exists():
            inspect_rows.append({'case_id': case_id, 'split': row.split, 'status': 'missing'})
            continue
        img = sitk.ReadImage(str(prior_path))
        arr = sitk.GetArrayFromImage(img)
        inspect_rows.append({
            'case_id': case_id,
            'split': row.split,
            'status': 'ok',
            'shape_zyx': tuple(arr.shape),
            'unique_values': np.unique(arr).tolist(),
            'ich_prior_voxels': int((arr > 0).sum()),
        })
    display(pd.DataFrame(inspect_rows))
else:
    print('Skipped prior inspection.')


,case_id,split,status,shape_zyx,unique_values,ich_prior_voxels
0,phe_0001,train,ok,"(32, 512, 512)","[0, 1]",8734
1,phe_0002,test,ok,"(32, 512, 512)","[0, 1]",1351
2,phe_0004,train,ok,"(32, 512, 512)","[0, 1]",14098
3,phe_0005,train,ok,"(32, 512, 512)","[0, 1]",5509
4,phe_0006,train,ok,"(32, 512, 512)","[0, 1]",38815
5,phe_0008,train,ok,"(32, 512, 512)","[0, 1]",17165
6,phe_0009,train,ok,"(32, 512, 512)","[0, 1]",3949
7,phe_0011,train,ok,"(30, 512, 512)","[0, 1]",5161


## 4. Convert PHE-SICH to 2-channel nnU-Net raw dataset

Raw dataset moi se co:

```text
imagesTr/phe_xxxx_0000.nii.gz  # CT
imagesTr/phe_xxxx_0001.nii.gz  # ICH teacher prior mask
labelsTr/phe_xxxx.nii.gz       # manual PHE
```

In [6]:
OVERWRITE_RAW_DATASET = True

def write_binary_like_reference(src_path: Path, reference_path: Path, dst_path: Path):
    src_path = Path(src_path)
    reference_path = Path(reference_path)
    dst_path = Path(dst_path)
    ref = sitk.ReadImage(str(reference_path))
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    phe_src = Path(row.phe_mask_path)
    ich_prior_src = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
    assert ich_prior_src.exists(), f'Missing ICH prior for {case_id}: {ich_prior_src}'

    if row.split == 'test':
        img_dst = imagesTs / f'{case_id}_0000.nii.gz'
        prior_dst = imagesTs / f'{case_id}_0001.nii.gz'
        seg_dst = labelsTs / f'{case_id}.nii.gz'
    else:
        img_dst = imagesTr / f'{case_id}_0000.nii.gz'
        prior_dst = imagesTr / f'{case_id}_0001.nii.gz'
        seg_dst = labelsTr / f'{case_id}.nii.gz'

    shutil.copy2(img_src, img_dst)
    write_binary_like_reference(ich_prior_src, img_dst, prior_dst)
    write_binary_like_reference(phe_src, img_dst, seg_dst)
    records.append({
        'case_id': case_id,
        'split': row.split,
        'ct_channel': str(img_dst),
        'ich_prior_channel': str(prior_dst),
        'phe_label': str(seg_dst),
        'source_image': str(img_src),
        'source_phe_mask': str(phe_src),
        'source_ich_prior': str(ich_prior_src),
    })

dataset_json = {
    'channel_names': {'0': 'CT', '1': 'ICH_teacher_prior'},
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_df.to_csv(OUTPUT_ROOT / '02_14_nnunet_phe_ich_prior_conversion_manifest.csv', index=False)

print('Dataset dir:', DATASET_DIR)
print('imagesTr channel files:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs channel files:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
print('dataset.json:', DATASET_DIR / 'dataset.json')
display(conversion_df.groupby('split').size().rename('cases').reset_index())


Dataset dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_raw\Dataset114_PHESICH_PHE_ICHTeacherPrior
imagesTr channel files: 218
labelsTr: 109
imagesTs channel files: 22
labelsTs: 11
dataset.json: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_raw\Dataset114_PHESICH_PHE_ICHTeacherPrior\dataset.json


,split,cases
0,test,11
1,train,99
2,val,10


## 5. Helper call entrypoints

In [7]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 6. Plan and preprocess Dataset114

In [8]:
CONFIGURATION = '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run.')


>>> nnUNetv2_plan_and_preprocess -d 114 -c 3d_fullres --verify_dataset_integrity -npfp 2 -np 1
Fingerprint extraction...
Dataset114_PHESICH_PHE_ICHTeacherPrior
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer


Extracting dataset fingerprint: 100%|██████████| 109/109 [00:16<00:00,  6.65it/s]


Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [5.         0.50292969 0.50292969]. 
Current patch size: (16, 320, 320). 
Current median shape: [ 30.         497.08737864 497.08737864]
Attempting to find 3d_lowres config. 
Current spacing: [5.         0.51801758 0.51801758]. 
Current patch size: (16, 320, 320). 
Current median shape: [ 30.         482.60910548 482.60910548]
Attempting to find 3d_lowres config. 
Current spacing: [5.         0.53355811 0.53355811]. 
Current patch size: (20, 320, 256). 
Current median shape: [ 30.         468.55252959 468.55252959]
Attempting to find 3d_lowres config. 
Current spacing: [5.         0.54956485 0.54956485]. 
Current patch size: (20, 32

Preprocessing cases: 100%|██████████| 109/109 [02:53<00:00,  1.60s/it]


Done in 3.6 min


## 7. Write manual fold-0 split

In [9]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_preprocessed\Dataset114_PHESICH_PHE_ICHTeacherPrior\splits_final.json
train: 99 val: 10 test: 11


## 8. Train PHE model with ICH prior

Day la phan train moi cho PHE. ICH teacher van giu nguyen, khong train lai.

In [10]:
RUN_TRAIN = True
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    run_training(
        str(DATASET_ID),
        CONFIGURATION,
        str(FOLD),
        trainer_class_name=TRAINER,
        plans_identifier=PLANS,
        export_validation_probabilities=EXPORT_VAL_NPZ,
        continue_training=CONTINUE_TRAINING,
        device=device,
    )
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


Training device: cuda
nnUNet_n_proc_DA: 0
nnUNet_compile: False

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-17 16:49:46.400275: do_dummy_2d_data_aug: True
2026-06-17 16:49:46.401269: Using splits from existing split file: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_preprocesse

## 9. Predict test set and evaluate PHE

In [11]:
RUN_PHE_PREDICT = True
RUN_PHE_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'

imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

print('Model dir        :', MODEL_DIR)
print('Fold dir         :', FOLD_DIR)
print('Using checkpoint :', RESOLVED_CHECKPOINT)

if RUN_PHE_PREDICT:
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('PHE predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PHE_PREDICT=True to run inference.')

if RUN_PHE_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_PHE_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


Model dir        : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_results\Dataset114_PHESICH_PHE_ICHTeacherPrior\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
Fold dir         : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_results\Dataset114_PHESICH_PHE_ICHTeacherPrior\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres\fold_0
Using checkpoint : checkpoint_best.pth
PHE predict device: cuda
There are 11 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 11 cases that I would like to predict

Predicting phe_0002:
perform_everything_on_device: True


100%|██████████| 27/27 [00:13<00:00,  1.95it/s]


sending off prediction to background worker for resampling and export
done with phe_0002

Predicting phe_0029:
perform_everything_on_device: True


100%|██████████| 12/12 [00:05<00:00,  2.14it/s]


sending off prediction to background worker for resampling and export
done with phe_0029

Predicting phe_0033:
perform_everything_on_device: True


...

100%|██████████| 27/27 [00:13<00:00,  1.99it/s]


sending off prediction to background worker for resampling and export
done with phe_0116

Predicting phe_0167:
perform_everything_on_device: True


100%|██████████| 12/12 [00:05<00:00,  2.13it/s]


sending off prediction to background worker for resampling and export
done with phe_0167
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
>>> nnUNetv2_evaluate_folder D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_raw\Dataset114_PHESICH_PHE_ICHTeacherPrior\labelsTs D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best -djfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_raw\Dataset114_PHESICH_PHE_ICHTeacherPrior\dataset.json -pfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\nnunet_workdir\nnUNet_preprocessed\Dataset114_PHESICH_PHE_ICHTeacherPrior\nnUNetPlans.json -o D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\summary.json -np 2
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Done in 0.0 min
Prediction dir: D:\Thuy_Loi\Nam_3\CT_xu

## 10. Export compact metrics CSV

In [12]:
def strip_nii_gz_name(path_like):
    name = Path(str(path_like)).name
    return name[:-7] if name.endswith('.nii.gz') else Path(name).stem

if SUMMARY_JSON.exists():
    with open(SUMMARY_JSON, 'r', encoding='utf-8') as f:
        summary = json.load(f)

    rows = []
    for item in summary.get('metric_per_case', []):
        metrics = item.get('metrics', {}).get('1', {})
        rows.append({
            'case_id': strip_nii_gz_name(item.get('prediction_file', '')),
            'prediction_file': item.get('prediction_file', ''),
            'reference_file': item.get('reference_file', ''),
            **metrics,
        })
    metrics_df = pd.DataFrame(rows)
    metrics_csv = OUTPUT_ROOT / '02_14_phe_ich_prior_metrics_per_case.csv'
    metrics_df.to_csv(metrics_csv, index=False)

    summary_rows = []
    numeric_cols = [c for c in metrics_df.columns if c not in {'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(metrics_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = OUTPUT_ROOT / '02_14_phe_ich_prior_metrics_summary.csv'
    summary_df.to_csv(summary_csv, index=False)

    print('Foreground mean from nnU-Net summary:')
    print(json.dumps(summary.get('foreground_mean', {}), indent=2))
    print('Wrote:', metrics_csv)
    print('Wrote:', summary_csv)
    display(summary_df)
else:
    print('No summary yet:', SUMMARY_JSON)
    print('Run train -> predict -> evaluate first.')


Foreground mean from nnU-Net summary:
{
  "Dice": 0.3998904803048216,
  "FN": 1033.4545454545455,
  "FP": 1876.0,
  "IoU": 0.2749941751070844,
  "TN": 7836160.7272727275,
  "TP": 1418.5454545454545,
  "n_pred": 3294.5454545454545,
  "n_ref": 2452.0
}
Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\02_14_phe_ich_prior_metrics_per_case.csv
Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14_nnunet_phe_ich_teacher_prior\02_14_phe_ich_prior_metrics_summary.csv


,label,metric,mean,median,std,n
0,PHE,Dice,3.998905e-01,5.021277e-01,0.226828,11
1,PHE,FN,1.033455e+03,8.650000e+02,1042.656604,11
2,PHE,FP,1.876000e+03,1.818000e+03,1064.383817,11
3,PHE,IoU,2.749942e-01,3.352273e-01,0.178041,11
4,PHE,TN,7.836161e+06,8.382624e+06,800994.682914,11
5,PHE,TP,1.418545e+03,9.240000e+02,1356.074908,11
6,PHE,n_pred,3.294545e+03,2.660000e+03,1722.882224,11
7,PHE,n_ref,2.452000e+03,1.635000e+03,2110.821512,11


## Notes

- `02_13` tao ICH teacher va danh gia ICH tren external test.
- `02_14` load ICH teacher do, predict ICH tren PHE-SICH, roi train PHE model voi `CT + ICH_teacher_prior`.
- De so sanh cong bang voi `02_12`, giu cung split va cung test set 11 case.